In [8]:
!pip install pillow_heif

  Using cached pillow_heif-0.18.0-cp311-cp311-win_amd64.whl.metadata (10 kB)
Using cached pillow_heif-0.18.0-cp311-cp311-win_amd64.whl (8.5 MB)


In [10]:
import os
from classes import BatchJobCreator, ProcessedBatch
from utils import *
from datetime import datetime
import logging

logger = logging.getLogger('classificador_comodos')

In [11]:
format_list = ['jpeg', 'jpg', 'png', '.heic']
jobs = []
model = 'gpt-4o-mini'
resolution = 'low'

In [12]:
base_directory = 'photos'

In [13]:
processed_dirs = load_processed_directories()

In [14]:
directory_list = os.listdir(base_directory)

In [15]:
directory_list = [folder_name for folder_name in directory_list if os.path.isdir(f'{base_directory}/{folder_name}') and folder_name not in processed_dirs]

In [16]:
directory_list

[]

In [17]:
for directory in directory_list:
    if directory in processed_dirs:
        continue

    base_path = f'{base_directory}/{directory}'
    image_list = os.listdir(base_path)

    for index, image_path in enumerate(image_list):
        image_name = '.'.join(image_path.split('/')[-1].split('.')[:-1])
        image_format = image_path.split('/')[-1].split('.')[-1]

        if image_format not in format_list:
            continue

        os.rename(f'{base_path}/{image_path}', f"{base_path}/{image_path.replace(image_name, f'{index:03}')}")

    image_list = os.listdir(base_path)
    image_list = [f'{base_directory}/{directory}/{file_name}' for file_name in image_list if file_name.split('.')[-1] in format_list]

    batchCreator = BatchJobCreator(directory, image_list, resolution, model)
    jobs.append(batchCreator)
    batchCreator.make_batch_request()

    save_batch({
        'timestamp' : datetime.now().isoformat(),
        'folder_id' : directory,
        'batch_id' : batchCreator.batch_id,
        'status' : 'processing',
        'model' : model,
    })


In [18]:
processed_batches = load_not_finished_batches()

In [19]:
processed_batches

[{'folder_id': '1PKtKomaHm',
  'batch_id': 'batch_66fdf308d9888190845d8559c121a0d0',
  'status': 'processing',
  'model': 'gpt-4o-mini',
  'timestamp': Timestamp('2024-10-02 22:27:37.225470'),
  'comment': nan},
 {'folder_id': 'S8muYVL10t',
  'batch_id': 'batch_66fdf4c26bbc8190a63ec34e7197025d',
  'status': 'processing',
  'model': 'gpt-4o-mini',
  'timestamp': Timestamp('2024-10-02 22:34:58.992432'),
  'comment': nan}]

In [20]:
batches = []

for batch in processed_batches:
    batch_to_check = ProcessedBatch(**batch)
    batches.append(batch_to_check)

    if not batch_to_check._check_for_results():
        print(batch_to_check.folder_id, batch_to_check.status)
        continue

    if len(batch_to_check.output_data) == 0 and len(batch_to_check.error_data) > 0:
        print('There are only errors.')
        batch_to_check.register_completed_batch('There are only data errors.')
        continue

    batch_to_check.rename_files()

    convert_heic_files_to_png(f'photos/{batch_to_check.folder_id}')
    add_water_mark(f'photos/{batch_to_check.folder_id}')

    batch_to_check.register_completed_batch()

2024-10-02 23:12:02,818 [classificador_comodos] [ERROR] - classes: Request error: 401 - b'{\n  "error": {\n    "message": "Incorrect API key provided: None. You can find your API key at https://platform.openai.com/account/api-keys.",\n    "type": "invalid_request_error",\n    "param": null,\n    "code": "invalid_api_key"\n  }\n}'
2024-10-02 23:12:02,820 [classificador_comodos] [INFO] - No HEIC files to convert
2024-10-02 23:12:11,920 [classificador_comodos] [ERROR] - classes: Request error: 401 - b'{\n  "error": {\n    "message": "Incorrect API key provided: None. You can find your API key at https://platform.openai.com/account/api-keys.",\n    "type": "invalid_request_error",\n    "param": null,\n    "code": "invalid_api_key"\n  }\n}'
2024-10-02 23:12:11,921 [classificador_comodos] [INFO] - No HEIC files to convert


Várias requisições para testar vários modelos

In [21]:
quality_options = ['low', 'high']
models = ['gpt-4o-mini', 'gpt-4o']

In [6]:
jobs = []

for model in models:
    for quality in quality_options:
        batchCreator = BatchJobCreator(image_list, quality, model)
        batchCreator.make_batch_request()
        jobs.append(batchCreator)

In [ ]:
for job in jobs:
    logger.info('-----------------------------------------------------')
    if not job._check_for_results():
        continue
    logger.info(f'model: {job.model}, quality: {job.quality}')
    job._compare_results()
    logger.info(f'precision: {job.precision}')
    logger.info(f'token_consuption: {job.get_total_spent_tokens()}, cost: {job.get_cost()}')
    logger.info('-----------------------------------------------------')
